# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianLogisticRegression, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[-0.92876683  0.05582122 -0.58349348 -0.28783714 -0.8966485 ]
 [-0.20389136  0.53501447  0.08730569  0.97876676 -0.19363073]
 [-0.10098811  0.61506803 -0.83110463 -0.92443323  0.19370602]
 [-0.12428422 -0.39279267 -0.86393976  0.47128589  0.73063053]
 [ 0.38975941  0.54023906  0.68912426 -0.72885236  0.77017637]
 [-0.97253775 -0.47727103  0.9821281  -0.52267268 -0.8894868 ]
 [ 0.51531355 -0.99111252  0.69421245  0.82597272 -0.27382573]
 [-0.52154508  0.72453345 -0.25444594  0.77867798  0.57717577]
 [-0.70704139 -0.23380128  0.66621025  0.79334216  0.42039906]
 [-0.31770731  0.67019345 -0.3110712   0.01805315  0.2958992 ]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_method = "VI"
update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianLogisticRegression(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianLogisticRegression(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a1', 'a1', 'a1', 'a2', 'a2', 'a1', 'a1', 'a1', 'a2', 'a2']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [0, 0, 0, 1, 1, 1, 1, 0, 1, 0]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/34 [00:00<?, ?it/s]

SVI:   3%|▎         | 1/34 [00:01<00:35,  1.09s/it]

SVI:   3%|▎         | 1/34 [00:01<00:35,  1.09s/it, loss=1213.0873]

SVI:   6%|▌         | 2/34 [00:01<00:34,  1.09s/it, loss=1183.5804]

SVI:   9%|▉         | 3/34 [00:01<00:33,  1.09s/it, loss=1096.8591]

...

SVI:  91%|█████████ | 31/34 [00:00<00:02,  1.18it/s, loss=741.3914]

SVI:  94%|█████████▍| 32/34 [00:00<00:01,  1.18it/s, loss=713.0710]

SVI:  97%|█████████▋| 33/34 [00:00<00:00,  1.18it/s, loss=711.6688]

SVI: 100%|██████████| 34/34 [00:01<00:00, 24.24it/s, loss=711.6688]

SVI: 100%|██████████| 34/34 [00:01<00:00, 24.24it/s, loss=670.7103]